Import all necessary libraries, covering parsing logs,labeling, analysis, recons

In [2]:
import pandas as pd
from pathlib import Path

# -------- parsing ----------
from parsing.syslog import parse_syslog_csv
from parsing.auditd import parse_audit_log
from parsing.auth import parse_auth_log
from parsing.zeek import (
    parse_conn_log, parse_dns_log, parse_http_log, 
    parse_files_log, parse_ssl_log, parse_ssh_log
)
from parsing.suricata import parse_suricata_eve
from parsing.azure_wins import (
    parse_azure_conn, parse_azure_process, parse_azure_security,
    parse_azure_events, parse_azure_port, parse_azure_perf
)
from parsing.tracee import parse_tracee_ndjson

# ---------- labeling / analysis / recons ----------
from labeling.tagger import tag_steps
from analysis.coverage import step_coverage
from analysis.failure_taxonomy import failure_taxonomy 
from analysis.metrics import compute_metrics
from analysis.ambig import ambiguity

from recons.event_graph import build_event_graph
from recons.chain_builder import reconstruct_chain
# --- scenarios config ---
from scenarios import SCENARIOS

LOG_ROOT = Path("logs")


define the mapping from source -> parser

In [ ]:
from typing import Callable, List, Tuple

def _find_files(log_root: Path, patterns: List[str]) -> List[Path]:
    ''' recursively find files matching any of the given patterns in log_root

    '''
    hits = []
    for pat in patterns:
        hits.extend(log_root.rglob(pat) if not pat.startswith("**/") else log_root.rglob(pat[3:]))
    # remove duplicates while preserving order
    uniq = sorted(set([p for p in hits if p.is_file()]))
    return uniq


def _parse_many(files: List[Path], parser: Callable[[str], pd.DataFrame], *, source: str, source_kind: str) -> pd.DataFrame:
    ''' Multiple files are parsed and merged using the same parser, and source fields are added.
    
    '''
    frames = []
    for p in files:
        try:
            df = parser(str(p))
        except Exception:
            print(f"Warning: failed to parse {p} with {parser.__name__}")
            continue
        if df is None or len(df) == 0:
            continue
        df = df.copy()
        df["source"] = source
        df["source_kind"] = source_kind
        df["source_file"] = str(p)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def parse_source(source_key: str, sc_log_root: Path) -> pd.DataFrame:
    '''
    return unified source dataframe for a given source_key
    args:
        sc_log_root: Path to scenario log root
    '''
    # --- syslog ---
    if source_key == "syslog":
        files = _find_files(sc_log_root, ["syslog.csv", "syslog*.csv"])
        return _parse_many(files, parse_syslog_csv, source="syslog", source_kind="syslog")
    
    # --- auditd ---
    if source_key == "auditd":
        files = _find_files(sc_log_root, ["audit.log", "auditd*.log"])
        return _parse_many(files, parse_audit_log, source="auditd", source_kind="auditd")

    # --- auth ---
    if source_key == "auth":
        files = _find_files(sc_log_root, ["auth.log", "auth*.log"])
        return _parse_many(files, parse_auth_log, source="auth", source_kind="auth")

    # ----- suricate -----
    if source_key == "suricata":
        files = _find_files(sc_log_root, ["eve.json", "eve*.json"])
        return _parse_many(files, parse_suricata_eve, source="suricata", source_kind="suricata")

    # --- zeek (multi-log) ---
    if source_key == "zeek":
        parts: List[Tuple[str, Callable[[str], pd.DataFrame]]] = [
            ("conn.log",  parse_conn_log),
            ("dns.log",   parse_dns_log),
            ("http.log",  parse_http_log),
            ("files.log", parse_files_log),
            ("ssh.log",   parse_ssh_log),
            ("ssl.log",   parse_ssl_log),
        ]
        frames = []
        for fname, fn in parts:
            files = _find_files(sc_log_root, [fname, fname.replace(".log", ".log.*"), fname.replace(".log", "*.log")])
            df = _parse_many(files, fn, source="zeek", source_kind=f"zeek_{fname.replace('.log','')}")
            if len(df):
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # --- azure windows ---
    if source_key == "azure_wins":
        

    return pd.DataFrame()